# Sensitivity Analysis — Shift Workload-Balancing Simulator

**SYNTHETIC DATA — FOR SIMULATION ONLY.**

Question: how sensitive are the balancing results to three key assumptions?

1. **Patient workload** — volume multiplier x1.0 (Normal), x1.1, x1.2, x1.3
2. **Nurse availability** — 100%, 90%, 80%, 70% of the Normal Day roster
3. **Acuity -> hours mapping** — "standard" (1,2,4,6,8 h) vs "demanding" (1.5,3,5,7.5,10 h)

plus a two-factor grid (patient volume x nurse availability).

All three sweeps are run through the real scenario + sequential-balancing pipeline
(`src.scenarios.run_scenario`), on top of the **Normal Day** scenario. Each level is repeated
for several random seeds (patient resampling / which nurses are absent differs per seed) and
reported as **mean ± standard deviation**, so one lucky or unlucky draw does not drive a
conclusion. Nothing below is invented — every number comes from running the simulation in
this notebook.

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

from config import (SENSITIVITY_PATIENT_MULTIPLIERS, SENSITIVITY_ABSENCE_LEVELS,
                    SENSITIVITY_ACUITY_MAPPINGS, CHARTS_DIR, RESULTS_DIR)
from src.scenarios import load_base_data
from src.sensitivity import run_sensitivity, run_grid, save_results
from src import charts

os.makedirs(CHARTS_DIR, exist_ok=True)
base_data = load_base_data()
print("Patient volume multipliers:", SENSITIVITY_PATIENT_MULTIPLIERS)
print("Nurse absence levels (-> availability %):", [f"{round((1-a)*100)}%" for a in SENSITIVITY_ABSENCE_LEVELS])
print("Acuity mappings:", SENSITIVITY_ACUITY_MAPPINGS)


Patient volume multipliers: [1.0, 1.1, 1.2, 1.3]
Nurse absence levels (-> availability %): ['100%', '90%', '80%', '70%']
Acuity mappings: ['standard', 'demanding']


## 1. Run the Three One-Factor Sweeps

`N_SEEDS` controls how many random seeds each level is repeated for. The notebook default
(5) runs in well under a minute; `run_experiments.py` (no `--quick`) uses 10 for the saved
`outputs/results/` files.

In [2]:
N_SEEDS = 5
raw, summary = run_sensitivity(base_data, n_seeds=N_SEEDS)
print(f"{len(raw)} individual scenario runs across all three dimensions.")
summary[["dimension", "level", "overloaded_before_mean", "overloaded_after_mean",
        "avg_ratio_after_mean", "max_ratio_after_mean", "transfers_mean", "rejected_mean",
        "safety_violations_total", "n_seeds"]].round(3)


50 individual scenario runs across all three dimensions.


,dimension,level,overloaded_before_mean,overloaded_after_mean,avg_ratio_after_mean,max_ratio_after_mean,transfers_mean,rejected_mean,safety_violations_total,n_seeds
0,Patient workload,Normal,6.0,1.0,0.910,1.056,6.0,22.0,0,5
1,Patient workload,+10%,8.0,3.0,0.973,1.025,16.0,44.0,0,5
2,Patient workload,+20%,9.4,8.8,1.057,1.359,17.8,101.6,0,5
3,Patient workload,+30%,13.8,7.6,1.054,1.329,50.4,95.2,0,5
4,Nurse availability,100%,6.0,1.0,0.910,1.056,6.0,22.0,0,5
5,Nurse availability,90%,8.4,3.4,0.982,1.116,17.4,40.4,0,5
6,Nurse availability,80%,11.8,10.0,1.061,1.285,22.6,102.2,0,5
7,Nurse availability,70%,14.6,13.6,1.176,1.531,27.8,114.2,0,5
8,Acuity mapping,Standard,6.0,1.0,0.910,1.056,6.0,22.0,0,5
9,Acuity mapping,Demanding,15.0,10.0,1.066,1.367,52.0,117.0,0,5


## 2. Patient Workload Sensitivity

In [3]:
display(summary[summary["dimension"] == "Patient workload"].round(3))
charts.plot_sensitivity(summary, "Patient workload", "sensitivity_patient_workload.png")
plt.show()


,dimension,level,level_order,overloaded_before_mean,overloaded_after_mean,avg_ratio_before_mean,avg_ratio_after_mean,max_ratio_before_mean,max_ratio_after_mean,mad_before_mean,...,mad_after_std,gap_before_std,gap_after_std,transfers_std,rejected_std,screened_out_std,safety_violations_std,mad_reduction_pct_std,safety_violations_total,n_seeds
0,Patient workload,Normal,0,6.0,1.0,0.928,0.910,1.071,1.056,0.119,...,0.000,0.0,0.0,0.000,0.000,0.000,0.0,0.000,0,5
1,Patient workload,+10%,1,8.0,3.0,1.021,0.973,1.204,1.025,0.117,...,0.003,0.0,0.0,2.608,7.694,84.930,0.0,5.306,0,5
2,Patient workload,+20%,2,9.4,8.8,1.118,1.057,1.311,1.359,0.154,...,0.011,0.0,0.0,3.250,12.339,60.487,0.0,7.019,0,5
3,Patient workload,+30%,3,13.8,7.6,1.203,1.054,1.426,1.329,0.207,...,0.008,0.0,0.0,3.929,12.368,43.060,0.0,1.628,0,5


## 3. Nurse Availability Sensitivity

In [4]:
display(summary[summary["dimension"] == "Nurse availability"].round(3))
charts.plot_sensitivity(summary, "Nurse availability", "sensitivity_nurse_availability.png")
plt.show()


,dimension,level,level_order,overloaded_before_mean,overloaded_after_mean,avg_ratio_before_mean,avg_ratio_after_mean,max_ratio_before_mean,max_ratio_after_mean,mad_before_mean,...,mad_after_std,gap_before_std,gap_after_std,transfers_std,rejected_std,screened_out_std,safety_violations_std,mad_reduction_pct_std,safety_violations_total,n_seeds
4,Nurse availability,100%,0,6.0,1.0,0.928,0.910,1.071,1.056,0.119,...,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0,5
5,Nurse availability,90%,1,8.4,3.4,1.053,0.982,1.484,1.116,0.148,...,0.009,1.600,0.400,4.409,6.375,94.956,0.0,5.609,0,5
6,Nurse availability,80%,2,11.8,10.0,1.164,1.061,1.574,1.285,0.192,...,0.005,1.625,0.000,3.323,9.108,28.703,0.0,7.472,0,5
7,Nurse availability,70%,3,14.6,13.6,1.335,1.176,2.098,1.531,0.341,...,0.013,4.363,2.245,4.354,11.089,72.736,0.0,5.482,0,5


## 4. Acuity Mapping Sensitivity (standard vs demanding)

In [5]:
display(summary[summary["dimension"] == "Acuity mapping"].round(3))
charts.plot_sensitivity(summary, "Acuity mapping", "sensitivity_acuity_mapping.png")
plt.show()


,dimension,level,level_order,overloaded_before_mean,overloaded_after_mean,avg_ratio_before_mean,avg_ratio_after_mean,max_ratio_before_mean,max_ratio_after_mean,mad_before_mean,...,mad_after_std,gap_before_std,gap_after_std,transfers_std,rejected_std,screened_out_std,safety_violations_std,mad_reduction_pct_std,safety_violations_total,n_seeds
8,Acuity mapping,Standard,0,6.0,1.0,0.928,0.910,1.071,1.056,0.119,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,5
9,Acuity mapping,Demanding,1,15.0,10.0,1.219,1.066,1.402,1.367,0.219,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,5


## 5. Two-Factor Grid: Patient Volume x Nurse Availability

Mean number of overloaded unit-shifts (of 15), before and after balancing, over a grid of
patient-volume multiplier x nurse availability. Fewer seeds per cell (grid has 16 cells) keep
this fast; `run_experiments.py` uses more.

In [6]:
grid = run_grid(base_data, n_seeds=3)
display(grid[["patient_multiplier", "availability", "overloaded_before_mean", "overloaded_after_mean",
             "avg_ratio_after_mean", "transfers_mean", "safety_violations_total"]].round(2))

charts.plot_sensitivity_heatmap(grid, "overloaded_before", "Overloaded unit-shifts BEFORE balancing (mean)", "sensitivity_heatmap_overloaded_before.png")
plt.show()
charts.plot_sensitivity_heatmap(grid, "overloaded_after", "Overloaded unit-shifts AFTER balancing (mean)", "sensitivity_heatmap_overloaded_after.png")
plt.show()


,patient_multiplier,availability,overloaded_before_mean,overloaded_after_mean,avg_ratio_after_mean,transfers_mean,safety_violations_total
0,1.0,100,6.00,1.00,0.91,6.00,0
1,1.0,90,8.67,3.00,0.98,19.67,0
2,1.0,80,12.00,10.00,1.06,23.67,0
3,1.0,70,14.67,13.33,1.18,30.67,0
4,1.1,100,8.00,2.67,0.97,14.33,0
5,1.1,90,10.67,8.33,1.07,14.67,0
6,1.1,80,13.67,9.67,1.12,42.33,0
7,1.1,70,14.67,13.33,1.27,32.33,0
8,1.2,100,9.67,9.00,1.07,16.67,0
9,1.2,90,14.00,9.33,1.08,52.00,0


## 6. Safety Check Across Every Sweep

In [7]:
total_violations = int(summary["safety_violations_total"].sum()) + int(grid["safety_violations_total"].sum())
print(f"Total safety violations across every sensitivity run (one-factor sweeps + grid): {total_violations}")
assert total_violations == 0, "A safety violation was found by the independent audit — investigate before trusting results."
print("No safety violation found in any run: the safety engine held under every tested assumption.")


Total safety violations across every sensitivity run (one-factor sweeps + grid): 0
No safety violation found in any run: the safety engine held under every tested assumption.


## 7. Save Results

In [8]:
save_results(raw, summary, grid)
print(f"Saved sensitivity_raw.csv, sensitivity_summary.csv, sensitivity_grid.csv to {RESULTS_DIR}")


Saved sensitivity_raw.csv, sensitivity_summary.csv, sensitivity_grid.csv to /home/claude/hospital_workload_simulator/outputs/results


## 8. Findings

- Both **more patients** and **fewer available nurses** increase the number of overloaded
  unit-shifts and the average/maximum workload ratio, as expected; the balancing engine reduces
  — but at the higher stress levels does not eliminate — overload, because fewer safe transfers
  exist when the whole hospital is short-staffed (see the "after" values trending up alongside
  the "before" values at the +20%/+30% and 80%/70% levels).
- The **acuity mapping** choice matters a great deal: the "demanding" mapping (roughly 25-50%
  more hours per patient) produces substantially more overloaded unit-shifts than "standard" at
  the same patient/nurse numbers, which is exactly the point of including it — it shows how much
  the result depends on an assumption that is explicitly documented as a simulation choice, not
  a measured clinical fact.
- No safety violation was found at any sweep level in Section 6: the safety engine's guarantees
  (never drop a source below minimum staffing, never recommend an unsafe/unqualified transfer)
  hold regardless of how stressed the simulated hospital is.

## 9. Limitations

- Each level above is averaged over a handful of seeds (5 for the one-factor sweeps, 3 for the
  grid in this notebook; `run_experiments.py` uses more by default) — enough to see the trend,
  but not a large-sample statistical study.
- The two-factor grid only covers patient volume x nurse availability; a full combination of
  all three dimensions (and more extreme levels) is part of the remaining 30% of the project.